# 🚀 AGAR-RL V6 : Pipeline SOTA Deep RL (Hunter-Predator & Magnetic Remerge)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Albin0903/agario/blob/main/notebooks/train_colab.ipynb)

**Entraînement de haute performance à pleine puissance (GPU L4 / A100) avec auto-sauvegarde Google Drive.**

### 🎯 Nouveautés Majeures de la V6 SOTA (AgarCL & GoBigger Standards) :
1. **Physique de Remerge Magnétique** : Correction fondamentale du moteur Agar.io. Dès expiration du cooldown, les sous-cellules sont magnétiquement attirées vers le centroïde même à pleine vitesse, garantissant la réunification rapide et éliminant la perte de cellules divisées.
2. **PBRS Chasse & Combat Prédateur Dominant** : Récompense continue d'approche des proies comestibles, bonus massif d'élimination (+25.0 par cellule ennemie dévorée), et modulation de la récolte de nourriture (les grosses cellules privilégient la traque des adversaires).
3. **Gating Tactique des Splits** : Récompense positive (+0.5) pour les divisions balistiques orientées vers une proie dans le cône de frappe (±45°, 50 < d < 320), et pénalité (-0.25) pour les divisions inutiles dans le vide.
4. **Reprise Transparente depuis V5 (13.75M steps)** : Chargement direct du checkpoint `ppo_latest.zip` de votre backup V5 pour pivoter immédiatement vers le comportement prédateur sans repartir de zéro.
5. **Inspecteur Diagnostic Intégré** : Sonde tactique automatisée (`src/analysis/inspect_policy.py`) pour analyser les réflexes du réseau de neurones.

## 0. Montage Google Drive & Détection GPU L4
Tous les checkpoints, les replays HD et les modèles ONNX seront automatiquement sauvegardés sur votre Drive dans le dossier `agario_rl_backup_v6`.

In [ ]:
# 1. Montage sécurisé de Google Drive
import os, sys, time, torch

try:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
except ImportError:
    pass

DRIVE_BACKUP_DIR = '/content/drive/MyDrive/agario_rl_backup_v6'
PREV_BACKUP_DIR = '/content/drive/MyDrive/agario_rl_backup_v5'
os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)

# 2. Vérification du matériel accéléré (GPU L4 / A100 recommandé)
print('=' * 65)
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'🚀 Accélération GPU Détectée : {gpu_name} ({vram:.1f} Go VRAM)')
    print('⚡ Configuration optimale : 16 environnements parallèles + Numba JIT + Batch 512')
else:
    print('⚠️ Aucun GPU détecté. Activez un GPU dans : Exécution > Modifier le type d\'exécution')
print(f'📁 Dossier Google Drive V6 synchronisé : {DRIVE_BACKUP_DIR}')
print(f'📁 Dossier précédent V5 disponible : {PREV_BACKUP_DIR} (Existe: {os.path.exists(PREV_BACKUP_DIR)})')
print('=' * 65)

## 1. Synchronisation du Code GitHub & Installation des Dépendances

In [ ]:
import os

# 1. Récupération propre des dernières modifications ou clonage
if os.path.exists('.git'):
    print('🔄 Synchronisation avec GitHub main...')
    !git fetch origin main
    !git reset --hard origin/main
elif os.path.exists('agario/.git'):
    print('🔄 Déplacement dans agario et synchronisation avec GitHub main...')
    %cd agario
    !git fetch origin main
    !git reset --hard origin/main
else:
    print('🌐 Clonage propre du repo...')
    !git clone https://github.com/Albin0903/agario.git
    %cd agario

# 2. Configuration du PYTHONPATH et installation des dépendances Farama Gymnasium
os.environ['PYTHONPATH'] = f"{os.getcwd()}:{os.environ.get('PYTHONPATH', '')}"
!pip uninstall -y -q gym 2>/dev/null || true
!pip install -q -r requirements.txt tensorboard
!apt-get install -qq -y ffmpeg
print('✅ Environnement et dépendances installés avec succès.')

## 2. Validation Pré-Vol : Suite Complète de 29 Tests Unitaires
Vérification complète de la physique de remerge magnétique, des récompenses de chasse, de la conformité Farama Gymnasium et de l'export ONNX.

In [ ]:
# Exécute tous les tests du moteur physique, du remerge et des récompenses Farama
!python -m pytest -v

## 3. Monitoring TensorBoard (Optionnel)

In [ ]:
import os
os.makedirs('logs/tensorboard', exist_ok=True)
try:
    %load_ext tensorboard
    %tensorboard --logdir logs/tensorboard
except Exception as e:
    print(f'Note TensorBoard : {e}')

## 4. Entraînement Haute Performance V6 (Hunter-Predator Architecture)
- **Reprise Optimale depuis V5** : Vous pouvez reprendre immédiatement à partir du checkpoint 13.75M steps stocké dans `agario_rl_backup_v5` pour que l'agent perfectionne la chasse sans réapprendre les bases du jeu !
- **Sauvegarde Continue V6** : Checkpoints automatiques tous les 250 000 pas dans `agario_rl_backup_v6`.

In [ ]:
# 🚀 Configuration de Reprise & Lancement V6
import os

# Activez RESUME_FROM_V5 = True pour transférer les 13.75M steps de votre checkpoint V5 existant
RESUME_FROM_V5 = True
V5_DIR = '/content/drive/MyDrive/agario_rl_backup_v5'
V6_DIR = '/content/drive/MyDrive/agario_rl_backup_v6'

resume_flag = "--resume auto"
if RESUME_FROM_V5:
    candidates = [
        os.path.join(V6_DIR, 'ppo_latest.zip'),
        os.path.join(V5_DIR, 'ppo_latest.zip'),
        os.path.join(V5_DIR, 'ppo_step_13750000.zip'),
    ]
    chosen = next((c for c in candidates if os.path.exists(c)), None)
    if chosen:
        resume_flag = f'--resume "{chosen}"'
        print(f'🎯 Checkpoint source détecté pour reprise : {chosen}')
    else:
        print('⚠️ Aucun checkpoint V5 trouvé, démarrage auto standard.')

!python src/training/train_colab.py \
    --n-envs 16 \
    --total-timesteps 15000000 \
    --pool-interval 250000 \
    --backup-dir {V6_DIR} \
    {resume_flag} \
    --device auto

## 5. Inspection Diagnostique de la Politique & Réflexes Tactiques
Sonde le réseau de neurones sur des scénarios synthétiques contrôlés : réponse à la nourriture, réflexe d'évitement des prédateurs, et propension à chasser/attaquer les proies en zone de frappe.

In [ ]:
import os, glob, re

def extract_step(path):
    if 'final' in os.path.basename(path):
        return 999999999
    m = re.search(r'step_(\d+)', path)
    return int(m.group(1)) if m else 0

candidates = glob.glob('/content/drive/MyDrive/agario_rl_backup_v6/*.zip') + \
             glob.glob('/content/drive/MyDrive/agario_rl_backup_v5/*.zip') + \
             glob.glob('checkpoints/ppo/*.zip')

candidates = [c for c in candidates if os.path.getsize(c) > 1000 and not os.path.basename(c).startswith('._') and 'bc_pretrained' not in c]
candidates.sort(key=extract_step, reverse=True)
target_inspect = candidates[0] if candidates else 'checkpoints/ppo/ppo_latest.zip'

print('=' * 75)
print(f'🔬 Inspection Diagnostique du Modèle : {target_inspect}')
print('=' * 75)

!python src/analysis/inspect_policy.py --model "{target_inspect}"

## 6. Enregistrement Automatique du Match Replay HD & Visualisation Directe
Génère une vidéo HD de 80 secondes (2400 steps @ 30 FPS) avec affichage tête haute (HUD), vecteurs de décision et radar.

In [ ]:
import os, glob, re
from IPython.display import HTML, display
from base64 import b64encode

def extract_step(path):
    if 'final' in os.path.basename(path):
        return 999999999
    m = re.search(r'step_(\d+)', path)
    return int(m.group(1)) if m else 0

candidates = glob.glob('/content/drive/MyDrive/agario_rl_backup_v6/*.zip') + \
             glob.glob('/content/drive/MyDrive/agario_rl_backup_v5/*.zip') + \
             glob.glob('checkpoints/ppo/*.zip')

valid_cands = [c for c in candidates if os.path.getsize(c) > 1000 and not os.path.basename(c).startswith('._') and 'bc_pretrained' not in c]
valid_cands.sort(key=extract_step, reverse=True)
target_model = valid_cands[0] if valid_cands else 'checkpoints/ppo/ppo_latest.zip'
step_count = extract_step(target_model)

print('=' * 75)
print(f'🎬 Modèle sélectionné pour le Replay HD : {target_model}')
print(f'📊 Palier : {step_count:,} steps')
print('=' * 75)

os.makedirs('recordings', exist_ok=True)
!python src/inference/record_match.py \
    --model "{target_model}" \
    --output recordings/eval_match_v6.mp4 \
    --steps 2400

if os.path.exists('recordings/eval_match_v6.mp4') and os.path.exists('/content/drive/MyDrive/agario_rl_backup_v6'):
    !cp recordings/eval_match_v6.mp4 /content/drive/MyDrive/agario_rl_backup_v6/eval_match_v6.mp4
    print('📁 Replay HD copié sur Google Drive dans : agario_rl_backup_v6/eval_match_v6.mp4')

video_path = 'recordings/eval_match_v6.mp4'
if os.path.exists(video_path):
    mp4_bytes = open(video_path, 'rb').read()
    data_url = 'data:video/mp4;base64,' + b64encode(mp4_bytes).decode()
    display(HTML(f'''
    <video width="850" height="480" controls autoplay loop>
        <source src="{data_url}" type="video/mp4">
    </video>
    '''))
    print(f'Taille de la vidéo : {os.path.getsize(video_path) / 1_000_000:.1f} Mo')
else:
    print('⚠️ Vidéo non trouvée.')

## 7. Exportation Universelle vers ONNX & Benchmark de Latence
Convertit le réseau de neurones PyTorch au standard ONNX ultra-rapide (< 0.02 ms de latence CPU).

In [ ]:
import os, glob, re

def extract_step(path):
    if 'final' in os.path.basename(path):
        return 999999999
    m = re.search(r'step_(\d+)', path)
    return int(m.group(1)) if m else 0

candidates = glob.glob('/content/drive/MyDrive/agario_rl_backup_v6/*.zip') + \
             glob.glob('/content/drive/MyDrive/agario_rl_backup_v5/*.zip') + \
             glob.glob('checkpoints/ppo/*.zip')

valid_cands = [c for c in candidates if os.path.getsize(c) > 1000 and not os.path.basename(c).startswith('._') and 'bc_pretrained' not in c]
valid_cands.sort(key=extract_step, reverse=True)
best_model = valid_cands[0] if valid_cands else None

if best_model and os.path.exists(best_model):
    print(f'Modèle sélectionné pour l\'export : {best_model}')
    os.makedirs('models', exist_ok=True)
    !python src/inference/export_onnx.py --model "{best_model}" --output models/model_v6.onnx
    if os.path.exists('/content/drive/MyDrive/agario_rl_backup_v6'):
        !cp models/model_v6.onnx /content/drive/MyDrive/agario_rl_backup_v6/model_v6.onnx
        print('📁 Modèle ONNX sauvegardé sur Drive : agario_rl_backup_v6/model_v6.onnx')
else:
    print('⚠️ Aucun checkpoint trouvé pour l\'export ONNX.')